Implementation of the Mamba Architecture

In [1]:
import tiktoken
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.nn.functional as F
import urllib.request as req

In [5]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

In [6]:
try:
    with req.urlopen(url) as response:
        raw_text = response.read().decode('utf-8')
    print("Total Length", len(raw_text))
except:
    print("Text download failed")

Total Length 1115394


In [11]:
corpus = raw_text[:150]
print("Corpus Excerpt: \n", "="*30, "\n", corpus)

Corpus Excerpt: 
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

A


In [15]:
enc = tiktoken.encoding_for_model("gpt-4o")
token_ids = enc.encode(raw_text)
vocab_size = enc.max_token_value + 1

In [16]:
data_tensor = torch.tensor(token_ids, dtype=torch.long)

Parameters

In [46]:
block_size = 128
batch_size = 16
d_model = 128
d_state = 16
n_layers = 4
eps = 1e-6
learning_rate = 1e-3
max_iters = 1000
eval_interval = 100
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Train/Val Split

In [21]:
n = int(0.9 * len(data_tensor))
train_data = data_tensor[:n]
val_data = data_tensor[n:]

Batching

In [24]:
def get_batch(split, device):
    data = train_data if split == "train" else val_data
    ix = torch.randint(0, len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

Verifying Batching

In [31]:
x, y = get_batch("train", device)
assert x.shape == (batch_size, block_size)
assert y.shape == (batch_size, block_size)

Root Mean Square Normalisation

In [33]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight

In [42]:
# Test RMS
rms = RMSNorm(d_model)
input_tensor = torch.randn([batch_size, block_size, d_model])
rms_test_res = rms(input_tensor)
assert input_tensor.shape == rms_test_res.shape

Mamba Block

In [44]:
class MambaBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
    
    def forward(self, x):
        return x

Tiny Mamba

In [48]:
class TinyMamba(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, eps):
        super().__init__()
        self.vocab_size = vocab_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([MambaBlock(d_model) for _ in range(n_layers)])
        self.rmsnorm = RMSNorm(d_model, eps)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx, targets):
        x = self.token_embedding(idx)
        for block in self.blocks:
            x = block(x)
        x = self.rmsnorm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            logits_flat = logits.view(-1, self.vocab_size)
            targets_flat = targets.view(-1)
            loss = F.cross_entropy(logits_flat, targets_flat)
        return logits, loss

In [49]:
# Test run
test_model = TinyMamba(vocab_size, d_model, n_layers, eps).to(device)
test_optimizer = torch.optim.AdamW(test_model.parameters(), lr=learning_rate)
test_logits, test_loss = test_model(x, y)
test_optimizer.zero_grad()
test_loss.backward()
test_optimizer.step()